In [2]:
import anndata
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import pandas as pd
from scipy.stats import spearmanr
from scvi.external import GIMVI
import scvi
import scipy.sparse as sparse
import seaborn as sns
from tqdm import tqdm

import json
from datetime import datetime

In [3]:
def X_is_raw(adata):
    return np.array_equal(adata.X.sum(axis=0).astype(int), adata.X.sum(axis=0))

In [4]:
timestamp = datetime.now().strftime('%d%m%Y_%H%M%S')

### Read in data

In [ ]:
seq_data = sc.read_h5ad('data/gut_data/Healthy_colon_adult.h5ad')

In [ ]:
spatial_data = sc.read_h5ad('data/gut_data/gut_hs_XeniumAdultColonNicheCompass_AM_21102024_150114_raw.h5ad')

In [7]:
#seq_data = seq_data[np.random.choice(seq_data.obs_names, int(0.2 * seq_data.shape[0]), replace=False)].copy()
#spatial_data = spatial_data[np.random.choice(spatial_data.obs_names, int(0.2 * spatial_data.shape[0]), replace=False)].copy()
#print(f"Sampled seq_data shape: {seq_data.shape}")
#print(f"Sampled spatial_data shape: {spatial_data.shape}")

## Identify common genes 

In [8]:
common_genes = list(set(spatial_data.var_names).intersection(set(seq_data.var_names)))

In [9]:
seq_data_common = seq_data[:, common_genes].copy()
spatial_data_common = spatial_data[:, common_genes].copy()

##  Setup data for gimVI

In [10]:
n_genes = len(common_genes)
train_frac = 0.95  # Train on 80% of common genes
n_train_genes = int(n_genes * train_frac)

In [11]:
np.random.seed(42)
train_gene_idx = np.random.choice(range(n_genes), n_train_genes, replace=False)
test_gene_idx = sorted(set(range(n_genes)) - set(train_gene_idx))

In [12]:
train_genes = [common_genes[i] for i in train_gene_idx]
test_genes = [common_genes[i] for i in test_gene_idx]

In [13]:
print(f"Training on {len(train_genes)} genes, holding out {len(test_genes)} genes for validation")

Training on 386 genes, holding out 21 genes for validation


+ Create training data with subset of common genes

In [14]:
spatial_train = spatial_data_common[:, train_genes].copy()
seq_train = seq_data_common[:, train_genes].copy()

In [15]:
sc.pp.filter_cells(spatial_train, min_counts=1)
sc.pp.filter_cells(seq_train, min_counts=1)

+ Setup AnnData for training

In [16]:
GIMVI.setup_anndata(
    spatial_train, 
    labels_key="C_scANVI",
    batch_key="Donor_ID"
)

GIMVI.setup_anndata(
    seq_train, 
    labels_key="C_scANVI",
    batch_key="Donor_ID"
)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:187: UserWarning: Category 21 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(


+ Train GIMVI

In [17]:
model = GIMVI(
    seq_train, 
    spatial_train,
    n_latent=128
)

In [18]:
model.train(
    max_epochs=3,
    early_stopping=True,
    early_stopping_patience=2
)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scvi/external/gimvi/_model.py:213: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator=='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/lightn

Epoch 3/3: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:07<00:00, 22.70s/it, v_num=1]

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 3/3: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:07<00:00, 22.66s/it, v_num=1]


+ Evaluate model training

In [ ]:
training_stats = model.history["elbo_validation"][1:]  # Skip first epoch
plt.figure(figsize=(10, 4))
plt.plot(training_stats)
plt.xlabel('Epoch')
plt.title('Training convergence')
#plt.savefig('Repos/Fetal_stem_cells_analysis/5_integration_genes_with_spatial/figures/GIMVI_elbow.png')
plt.show()

* Evaluate imputation

In [ ]:
def evaluate_imputation(model, spatial_data_common, test_genes, train_genes):
    """Evaluate imputation performance on held-out test genes"""
    # Create a spatial test dataset with both train and test genes
    spatial_test_full = spatial_data_common[:, train_genes + test_genes].copy()
    
    # Use same cells as in training data
    spatial_test_full = spatial_test_full[spatial_train.obs_names].copy()
    
    # Get imputed values (normalized)
    _, imputed_values = model.get_imputed_values(normalized=True)
    
    # Compute correlation for each test gene
    correlations = []
    
    # For each test gene, compare actual vs imputed
    for i, gene in enumerate(test_genes):
        # Get index of gene in the original data
        idx_in_test = test_genes.index(gene)
        imputed_expr = imputed_values[:, len(train_genes) + idx_in_test]
        
        # Get true and imputed expression
        true_expr = spatial_test_full[:, gene].X.toarray().flatten()
        
        # Calculate correlation only if there's variation in both
        if np.std(true_expr) > 0 and np.std(imputed_expr) > 0:
            corr = spearmanr(true_expr, imputed_expr)[0]
        else:
            corr = 0  # No correlation if no variation
            
        correlations.append((gene, corr))
        
    return correlations

In [ ]:
correlations = evaluate_imputation(model, spatial_data_common, test_genes, train_genes)

In [ ]:
corr_values = [c[1] for c in correlations]
median_corr = np.median(corr_values)
print(f"Median Spearman correlation: {median_corr:.4f}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(corr_values, bins=20, kde=True)
plt.axvline(median_corr, color='red', linestyle='--', label=f'Median r = {median_corr:.4f}')
plt.xlabel("Spearman correlation")
plt.ylabel("Number of genes")
plt.title("Distribution of imputation accuracy")
plt.legend()
#plt.savefig("imputation_accuracy.png", dpi=300, bbox_inches='tight')
plt.show()

## Get latent representations

In [19]:
latent_seq, latent_spatial = model.get_latent_representation()

In [20]:
latent_combined = np.concatenate([latent_seq, latent_spatial])
latent_adata = anndata.AnnData(latent_combined)

In [21]:
latent_adata.obs["data_source"] = (["scRNA-seq"] * latent_seq.shape[0] + 
                                  ["Xenium"] * latent_spatial.shape[0])

In [22]:
if "C_scANVI" in seq_train.obs.columns:
    labels = list(seq_train.obs["C_scANVI"]) + list(spatial_train.obs["C_scANVI"])
    latent_adata.obs["cell_type"] = labels

In [ ]:
sc.pp.neighbors(latent_adata, use_rep="X")
sc.tl.umap(latent_adata)

In [ ]:
sc.pl.umap(latent_adata, color=["data_source", "cell_type"])

## Full gene imputation

In [23]:
sc.pp.filter_cells(seq_data_common, min_counts=1)
sc.pp.filter_cells(spatial_data_common, min_counts=1)

In [24]:
GIMVI.setup_anndata(
        spatial_data_common,
        labels_key="C_scANVI",
        batch_key="Donor_ID")
    
GIMVI.setup_anndata(
        seq_data_common,
        labels_key="C_scANVI",
        batch_key="Donor_ID")

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:187: UserWarning: Category 21 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(


In [25]:
full_model = GIMVI(
        seq_data_common,
        spatial_data_common,
        n_latent=128
    )

In [26]:
full_model.train(
        max_epochs=3,
        early_stopping=True,
        early_stopping_patience=10
    )

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scvi/external/gimvi/_model.py:213: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator=='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/lightn

Epoch 3/3: 100%|██████████████████████████████████████████████████████████████████████████████| 3/3 [01:10<00:00, 23.47s/it, v_num=1]

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 3/3: 100%|██████████████████████████████████████████████████████████████████████████████| 3/3 [01:10<00:00, 23.58s/it, v_num=1]


## Train SCVI on full sequencing dataset

In [ ]:
seq_data

In [27]:
seq_data_full = seq_data.copy()

In [ ]:
# Calculate the number of genes to keep (5% of total genes)
num_genes_to_keep = int(len(seq_data_full.var_names) * 0.05)
selected_gene_indices = np.random.choice(range(len(seq_data_full.var_names)), num_genes_to_keep, replace=False)
seq_data_full = seq_data_full[:, selected_gene_indices].copy()
print(seq_data_full.shape)

In [28]:
common_cells = [cell for cell in seq_data_common.obs_names if cell in seq_data_full.obs_names]
seq_data_full = seq_data_full[common_cells]

In [29]:
seq_data_full.layers['counts'] = seq_data_full.X.copy()

/var/folders/kr/nd4y_1_s34n42lrht8wdh4cr0000gq/T/ipykernel_50458/1974744974.py:1: ImplicitModificationWarning: Setting element `.layers['counts']` of view, initializing view as actual.
  seq_data_full.layers['counts'] = seq_data_full.X.copy()


In [30]:
scvi.model.SCVI.setup_anndata(
        seq_data_full,
        labels_key="C_scANVI",
        batch_key="Donor_ID",
        layer = 'counts')

In [31]:
scvi_model = scvi.model.SCVI(seq_data_full,
                            n_latent = 50, 
                            n_layers = 3, 
                            dispersion = 'gene-batch', 
                            gene_likelihood = 'nb')

In [32]:
scvi_model.train(3, 
                 check_val_every_n_epoch = 1, 
                 enable_progress_bar = True)

/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/scvi/train/_trainrunner.py:69: UserWarning: `accelerator` has been automatically set to `cpu` although 'mps' exists. If you wish to run on mps backend, use explicitly accelerator=='mps' in train function.In future releases it will become default for mps supported machines.
  accelerator, lightning_devices, device = parse_device_args(
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
/Users/annamaguza/Library/Application Support/hatch/env/virtual/single-cell-project/0fJ3OPid/single_cell_project/lib/python3.11/site-packages/

Epoch 3/3: 100%|██████████████████████████| 3/3 [09:46<00:00, 198.43s/it, v_num=1, train_loss_step=3.47e+3, train_loss_epoch=3.42e+3]

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 3/3: 100%|██████████████████████████| 3/3 [09:46<00:00, 195.51s/it, v_num=1, train_loss_step=3.47e+3, train_loss_epoch=3.42e+3]


In [33]:
seq_data_full.obsm["X_scVI"] = scvi_model.get_latent_representation(seq_data_full)

In [ ]:
latent_seq_scvi = scvi_model.get_latent_representation()

In [ ]:
history_df = (
    scvi_model.history['elbo_train'].astype(float)
    .join(scvi_model.history['elbo_validation'].astype(float))
    .reset_index()
    .melt(id_vars=['epoch'])
)
plt.figure(figsize=(12, 10))

plt.plot(history_df[history_df['variable'] == 'elbo_train']['epoch'], 
         history_df[history_df['variable'] == 'elbo_train']['value'], 
         color='black', marker='o', label='Training ELBO')

plt.plot(history_df[history_df['variable'] == 'elbo_validation']['epoch'],
         history_df[history_df['variable'] == 'elbo_validation']['value'], 
         color='red', marker='o', label='Validation ELBO')

plt.xlabel('Epoch')
plt.ylabel('ELBO Value')
plt.title('Training and Validation ELBO over Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.xlim(left=1)

plt.tight_layout()
plt.show()

In [ ]:
seq_cells_in_gimvi = model.adatas[0].obs_names
seq_data_full = seq_data_full[seq_data_full.obs_names.isin(seq_cells_in_gimvi)]

In [ ]:
seq_cells_in_both = list(set(seq_cells_in_gimvi).intersection(set(seq_data_full.obs_names)))

In [ ]:
gimvi_indices = [list(seq_cells_in_gimvi).index(cell) for cell in seq_cells_in_both]
scvi_indices = [list(seq_data_full.obs_names).index(cell) for cell in seq_cells_in_both]

In [ ]:
latent_gimvi_subset = latent_seq[gimvi_indices]
latent_scvi_subset = latent_seq_scvi[scvi_indices]

In [ ]:
from sklearn.linear_model import Ridge
latent_mapper = Ridge(alpha=1.0)
latent_mapper.fit(latent_gimvi_subset, latent_scvi_subset)

In [ ]:
spatial_in_scvi_latent = latent_mapper.predict(latent_spatial)

In [ ]:
common_genes = list(set(spatial_data.var_names).intersection(set(seq_data.var_names)))
unique_seq_genes = list(set(seq_data.var_names) - set(spatial_data.var_names))
print(f"Number of genes to impute: {len(unique_seq_genes)}")

In [ ]:
decoder = scvi_model.module.decoder

In [ ]:
all_genes = list(spatial_data.var_names) + unique_seq_genes
imputed_data = np.zeros((spatial_data.shape[0], len(all_genes)))

In [ ]:
for i, gene in enumerate(spatial_data.var_names):
    idx = all_genes.index(gene)
    imputed_data[:, idx] = spatial_data[:, gene].X.toarray().flatten()

In [ ]:
import torch
spatial_latent_tensor = torch.tensor(spatial_in_scvi_latent, dtype=torch.float32)

In [ ]:
batch_size = 128
n_batches = int(np.ceil(spatial_latent_tensor.shape[0] / batch_size))

In [ ]:
seq_data.obs

In [ ]:
temp_adata = anndata.AnnData(
    X=np.zeros((spatial_data.shape[0], seq_data_full.shape[1])),
    obs=pd.DataFrame(index=[f"spatial_{i}" for i in range(spatial_data.shape[0])]),
    var=seq_data_full.var.copy()
)

temp_adata.obsm["X_scvi"] = spatial_in_scvi_latent

temp_adata.layers["counts"] = temp_adata.X.copy()
temp_adata.obs["Donor_ID"] = 'N10'
imputed_expression = scvi_model.get_normalized_expression(
    adata=temp_adata,
    extend_categories = True,
    indices=list(range(temp_adata.n_obs)),
    transform_batch=None,
    gene_list=None,
    library_size="latent",  # Use latent library size
    n_samples=1,
    return_mean=True,
    return_numpy=True
)

all_genes = list(spatial_data.var_names) + unique_seq_genes
imputed_data = np.zeros((spatial_data.shape[0], len(all_genes)))

# Copy the measured values for common genes
for i, gene in enumerate(spatial_data.var_names):
    idx = all_genes.index(gene)
    imputed_data[:, idx] = spatial_data[:, gene].X.toarray().flatten()

# Map the imputed genes to the right positions
for i, gene in enumerate(seq_data_full.var_names):
    if gene in unique_seq_genes:
        gene_idx = all_genes.index(gene)
        imputed_data[:, gene_idx] = imputed_expression[:, i]


In [ ]:
temp_adata.obs

In [ ]:
imputed_adata = anndata.AnnData(
    X=imputed_data,
    obs=spatial_data.obs.copy(),
    var=pd.DataFrame(index=all_genes)
)